# Standard KITTI 3D LiDAR training (Google Colab)

Notebook dùng chung cho A0–A6 và B0–B3. Chỉ sửa cell **Configuration**; mỗi run được khóa theo branch, commit và config hash trước khi resume.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import os
import subprocess
import torch

# Default branch contains every registered A/B config. Change this for a proposal branch.
BRANCH = "A56_proposal_2.5"
VARIANT = "A0"  # A0–A6 or B0–B3
CONFIG_OVERRIDE = None  # e.g. "configs/kitti/my_proposal.json"
RUN_NAME = None  # None creates a stable, resume-friendly name
SEED = 42
PRECISION = "bf16"  # fp32, fp16, bf16 (BF16 requires a supported GPU)
PHYSICAL_BATCH_SIZE = 2
ACCUMULATION_STEPS = 2
EPOCHS = 100  # Safe to increase when resuming the same run
NUM_WORKERS = max(0, min(8, (os.cpu_count() or 1) - 1))
RUN_SMOKE_TEST = True
RUN_EVALUATION = True
ALLOW_LEGACY_RESUME = False  # True only for runs created before this notebook

REPOSITORY_URL = "https://github.com/danhyoyo/Lidar.git"
REPO_DIR = Path("/content/Lidar")
KITTI_TAR_ROOT = Path("/content/drive/MyDrive/KITTI_DATASET_ZIP")
RAW_KITTI_ROOT = Path("/content/KITTI_DATASET")
PROCESSED_DATASET_DIR = REPO_DIR / "data/kitti/processed"
ARTIFACT_ROOT = Path("/content/drive/MyDrive/lidar_training_artifacts")

drive.mount("/content/drive")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "--prune", "origin", f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", f"origin/{BRANCH}"], cwd=REPO_DIR, check=True)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print(f"Branch: {BRANCH}\nCommit: {COMMIT}")

if not torch.cuda.is_available():
    raise RuntimeError("Bật GPU trong Runtime > Change runtime type trước khi train.")
if PRECISION == "bf16" and not torch.cuda.is_bf16_supported():
    raise RuntimeError("GPU này không hỗ trợ BF16; đổi PRECISION thành fp16 hoặc fp32.")
print(f"PyTorch: {torch.__version__}; GPU: {torch.cuda.get_device_name(0)}")

## Dependencies and variant

`CONFIG_OVERRIDE` is the escape hatch for a new proposal; it must be a repository-relative JSON path.

In [ ]:
%cd /content/Lidar
%pip install -q shapely onnx tqdm

VARIANT_CONFIGS = {
    "A0": "configs/kitti/kitti_uwag_coordatt_aug.json",
    "A1": "configs/kitti/mobilebev/a1_legacy35_center3d.json",
    "A2": "configs/kitti/mobilebev/a2_rich8_center3d.json",
    "A3": "configs/kitti/mobilebev/a3_legacy35_sgfpn_center3d.json",
    "A4": "configs/kitti/mobilebev/a4_rich8_sgfpn_center3d.json",
    "A5": "configs/kitti/mobilebev/a5_rich11_center3d.json",
    "A6": "configs/kitti/mobilebev/a6_rich11_sgfpn_center3d.json",
    "B0": "configs/kitti/probgeo_uq/b0_deterministic.json",
    "B1": "configs/kitti/probgeo_uq/b1_gwd.json",
    "B2": "configs/kitti/probgeo_uq/b2_heteroscedastic.json",
    "B3": "configs/kitti/probgeo_uq/b3_probgeo_uq.json",
}
if CONFIG_OVERRIDE is None and VARIANT not in VARIANT_CONFIGS:
    raise ValueError(f"Unknown VARIANT={VARIANT!r}; use CONFIG_OVERRIDE for a new proposal.")
CONFIG_RELATIVE = CONFIG_OVERRIDE or VARIANT_CONFIGS[VARIANT]
CONFIG = (REPO_DIR / CONFIG_RELATIVE).resolve()
if REPO_DIR not in CONFIG.parents or not CONFIG.is_file():
    raise FileNotFoundError(f"Config unavailable on {BRANCH}: {CONFIG_RELATIVE}")
CONFIG_DATA = json.loads(CONFIG.read_text(encoding="utf-8"))
EFFECTIVE_BATCH_SIZE = PHYSICAL_BATCH_SIZE * ACCUMULATION_STEPS
RUN_NAME = RUN_NAME or f"{VARIANT.lower()}_seed{SEED}_eb{EFFECTIVE_BATCH_SIZE}"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Variant: {VARIANT}\nConfig: {CONFIG_RELATIVE}\nRun: {RUN_NAME}")

## Prepare KITTI

The cell is idempotent: complete folders are reused; incomplete folders are re-extracted and validated.

In [ ]:
archives = {"velodyne": ("*.bin", 7481), "label_2": ("*.txt", 7481), "calib": ("*.txt", 7481)}
RAW_KITTI_ROOT.mkdir(parents=True, exist_ok=True)
for folder, (pattern, expected_count) in archives.items():
    archive = KITTI_TAR_ROOT / f"{folder}.tar"
    target = RAW_KITTI_ROOT / "training" / folder
    if sum(1 for _ in target.glob(pattern)) != expected_count:
        if not archive.is_file():
            raise FileNotFoundError(f"Missing KITTI archive: {archive}")
        subprocess.run(["tar", "--no-same-owner", "-xf", str(archive), "-C", str(RAW_KITTI_ROOT)], check=True)
    actual_count = sum(1 for _ in target.glob(pattern))
    if actual_count != expected_count:
        raise RuntimeError(f"{folder}: found {actual_count}, expected {expected_count}")

pointcloud_dir = PROCESSED_DATASET_DIR / "pointcloud"
label_dir = PROCESSED_DATASET_DIR / "label"
dataset_ready = (sum(1 for _ in pointcloud_dir.glob("*.bin")) == 7481 and sum(1 for _ in label_dir.glob("*.txt")) == 7481 and (PROCESSED_DATASET_DIR / "train.txt").is_file() and (PROCESSED_DATASET_DIR / "val.txt").is_file())
if not dataset_ready:
    subprocess.run(["python3", "tools/kitti_training_pipeline/prepare_kitti.py", "--kitti-root", str(RAW_KITTI_ROOT), "--output-root", str(PROCESSED_DATASET_DIR), "--config-output", str(REPO_DIR / "data/kitti/generated_kitti.json"), "--train-ids", "splits/kitti/train.txt", "--val-ids", "splits/kitti/val.txt", "--pointcloud-mode", "symlink", "--overwrite"], cwd=REPO_DIR, check=True)
assert sum(1 for _ in pointcloud_dir.glob("*.bin")) == sum(1 for _ in label_dir.glob("*.txt")) == 7481
print(f"KITTI ready: {PROCESSED_DATASET_DIR}")

## Verify the checked-out proposal

Run this before the smoke/full training cells, so each branch proves its own test contract.

In [ ]:
%cd /content/Lidar
!MPLCONFIGDIR=/tmp/lidar-mpl python3 tests/test_mobile_bev.py
if _exit_code:
    raise RuntimeError("Proposal tests failed; do not train this checkout.")

## Smoke test

This uses a separate run directory, so it never contaminates a resumable full run.

In [ ]:
if RUN_SMOKE_TEST:
    subprocess.run(["python3", "tools/kitti_training_pipeline/train.py", "--config", str(CONFIG), "--detector-root", "detector", "--output-root", str(ARTIFACT_ROOT), "--run-name", f"{RUN_NAME}_smoke", "--device", "cuda", "--precision", PRECISION, "--seed", str(SEED), "--epochs", "1", "--physical-batch-size", str(PHYSICAL_BATCH_SIZE), "--accumulation-steps", str(ACCUMULATION_STEPS), "--max-train-batches", "8", "--max-val-batches", "4", "--num-workers", str(NUM_WORKERS)], cwd=REPO_DIR, check=True)

## Full training / safe resume

The trainer writes `checkpoints/last.pt` after every completed epoch. Interrupting the log tail does not stop the detached training process; rerun this cell to reconnect.

In [ ]:
import hashlib
import re

RUN_DIR = ARTIFACT_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
TRAIN_LOG = RUN_DIR / "train.log"
PID_PATH = RUN_DIR / "train.pid"
RUN_METADATA_PATH = RUN_DIR / "run.json"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_METADATA = {"branch": BRANCH, "commit": COMMIT, "variant": VARIANT, "config_relative": CONFIG_RELATIVE, "config_sha256": hashlib.sha256(CONFIG.read_bytes()).hexdigest(), "seed": SEED, "precision": PRECISION, "physical_batch_size": PHYSICAL_BATCH_SIZE, "accumulation_steps": ACCUMULATION_STEPS}
if RUN_METADATA_PATH.is_file():
    if json.loads(RUN_METADATA_PATH.read_text(encoding="utf-8")) != RUN_METADATA:
        raise RuntimeError("Run metadata differs; choose a new RUN_NAME instead of resuming incompatible state.")
elif any(CHECKPOINT_DIR.glob("*.pt")) and not ALLOW_LEGACY_RESUME:
    raise RuntimeError("Existing checkpoints have no run.json. Set ALLOW_LEGACY_RESUME=True only after verifying compatibility.")
else:
    RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2, sort_keys=True) + "\n", encoding="utf-8")

def is_training(pid):
    try:
        return b"tools/kitti_training_pipeline/train.py" in Path(f"/proc/{pid}/cmdline").read_bytes()
    except (FileNotFoundError, PermissionError):
        return False

def checkpoint_epoch(path):
    if not path.is_file():
        return -1
    state = torch.load(path, map_location="cpu", weights_only=False)
    return int(state.get("epoch", -1)) if isinstance(state, dict) else -1

last_checkpoint = CHECKPOINT_DIR / "last.pt"
epoch_checkpoints = [path for path in CHECKPOINT_DIR.glob("*epoch.pt") if re.fullmatch(r"\d+epoch\.pt", path.name)]
resume_checkpoint = last_checkpoint if last_checkpoint.is_file() else max(epoch_checkpoints, key=checkpoint_epoch, default=None)
resume_epoch = checkpoint_epoch(resume_checkpoint) if resume_checkpoint else 0
training_pid = int(PID_PATH.read_text()) if PID_PATH.is_file() else 0
if is_training(training_pid):
    print(f"Training is already running (PID {training_pid}).")
elif resume_epoch >= EPOCHS:
    print(f"Training already reached epoch {resume_epoch}/{EPOCHS}.")
    training_pid = 0
else:
    command = ["python3", "-u", "tools/kitti_training_pipeline/train.py", "--config", str(CONFIG), "--detector-root", "detector", "--output-root", str(ARTIFACT_ROOT), "--run-name", RUN_NAME, "--device", "cuda", "--precision", PRECISION, "--seed", str(SEED), "--epochs", str(EPOCHS), "--physical-batch-size", str(PHYSICAL_BATCH_SIZE), "--accumulation-steps", str(ACCUMULATION_STEPS), "--num-workers", str(NUM_WORKERS)]
    if resume_checkpoint:
        command += ["--resume", str(resume_checkpoint)]
        print(f"Resuming epoch {resume_epoch}: {resume_checkpoint}")
    with TRAIN_LOG.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(command, cwd=REPO_DIR, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
    training_pid = process.pid
    PID_PATH.write_text(str(training_pid), encoding="utf-8")
    print(f"Started PID {training_pid}; log: {TRAIN_LOG}")

if training_pid:
    try:
        subprocess.run(["tail", f"--pid={training_pid}", "-n", "20", "-f", str(TRAIN_LOG)], check=False)
    except KeyboardInterrupt:
        print("Log tail stopped; the detached training process continues. Rerun this cell to reconnect.")

## Select checkpoint and evaluate

Center3D models use 3D mAP on a validation/calibration split. Legacy BEV A0 uses the trainer's validation-loss selection. B2/B3 add uncertainty evaluation.

In [ ]:
def run(command):
    print("$", " ".join(map(str, command)))
    subprocess.run(command, cwd=REPO_DIR, check=True)

is_center3d = CONFIG_DATA["model"].get("box_encoding", "bev") == "center3d"
selection_split = REPO_DIR / ("splits/kitti/uq_calibration.txt" if VARIANT.startswith("B") else "splits/kitti/val.txt")
if is_center3d:
    SELECTED_DIR = RUN_DIR / "selected_3d"
    run(["python3", "tools/kitti_training_pipeline/select_checkpoint.py", "--checkpoint-dir", str(CHECKPOINT_DIR), "--config", str(CONFIG), "--detector-root", "detector", "--kitti-root", str(RAW_KITTI_ROOT), "--split", str(selection_split), "--output-dir", str(SELECTED_DIR), "--device", "cuda"])
    CHECKPOINT = SELECTED_DIR / "best.pt"
else:
    CHECKPOINT = RUN_DIR / "selected" / "best.pt"
if not CHECKPOINT.is_file():
    raise FileNotFoundError(CHECKPOINT)

if RUN_EVALUATION:
    evaluation_splits = [("validation", REPO_DIR / "splits/kitti/val.txt")] if not VARIANT.startswith("B") else [("calibration", REPO_DIR / "splits/kitti/uq_calibration.txt"), ("test", REPO_DIR / "splits/kitti/uq_test.txt")]
    outputs = {}
    for name, split in evaluation_splits:
        output = RUN_DIR / f"evaluation_{name}.json"
        run(["python3", "tools/kitti_training_pipeline/evaluate_kitti_bev.py", "--name", f"{RUN_NAME}_{name}", "--backend", "pytorch", "--model", str(CHECKPOINT), "--config", str(CONFIG), "--detector-root", "detector", "--kitti-root", str(RAW_KITTI_ROOT), "--split", str(split), "--output", str(output), "--device", "cuda", "--warmup-frames", "10"])
        outputs[name] = output
    if VARIANT in {"B2", "B3"}:
        run(["python3", "tools/kitti_training_pipeline/evaluate_uncertainty.py", "--predictions", str(outputs["test"].with_suffix(".predictions.npz")), "--config", str(CONFIG), "--kitti-root", str(RAW_KITTI_ROOT), "--split", str(REPO_DIR / "splits/kitti/uq_test.txt"), "--calibration-predictions", str(outputs["calibration"].with_suffix(".predictions.npz")), "--calibration-split", "splits/kitti/uq_calibration.txt", "--output", str(RUN_DIR / "uncertainty_test.json")])
print(f"Checkpoint: {CHECKPOINT}")